# Bibliotecas

In [ ]:
import sys
import os

sys.path.append(os.path.abspath('../'))

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import time
import json

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler

import src.futurai_ppd as ppd
import src.futurai_utils as utils


from src.MSCVAE import MSCVAE
from src.MSCRED import MSCRED
from src.ANN_AE import ANN_AE
from src.LSTM_AE import LSTM_AE
from src.CNN_AE import CNN_AE

DIR_DATA = "/Users/falcone/Desktop/github-repos/anomaly-detection/data/"

# Parâmetros gerais e dados do processo

In [ ]:
base_name = "392P2007 Bomba.csv"
timestamp = "Timestamp"
process_name = "392P2007 Bomba"
company_name = "Klabin - MA"
subsistema = DIR_DATA+ process_name +'_subsistema.csv'

# Carregando dados

In [ ]:
list_colomuns_drop = ['392M3820_ST', "392CT2150_MV"] #"392CT2150_MV"

df_dataset = ppd.load_dataset_principal(DIR_DATA+process_name+".csv", list_colomuns_drop, timestamp, dropna=True, use_chunks=True, chunksize=10000)
df_dataset

# Parâmetros de ON/OFF

In [ ]:
pre_process = []

pp_var_ref_desligado = '392P2007_ST'
pp_valor_ref_desligado = 400
pp_tempo_ref_desligado = 0
pp_pre_corte_transitorio = 90
pp_pos_corte_transitorio = 150
pre_process.append(  
{
   "after_cut": pp_pos_corte_transitorio,
   "interval_off": pp_tempo_ref_desligado,
   "limit_off": pp_valor_ref_desligado,
   "pre_cut": pp_pre_corte_transitorio,
   "variable_off": pp_var_ref_desligado
  })

# Importação das tags e descrições

In [ ]:
df_sistema, df_sistema_drop =  ppd.set_tags_config(df_dataset,subsistema)
df_sistema

In [ ]:
df_sistema_drop

# Seleção do período de treinamento e criação do modelo

In [ ]:
df_train_list = []

## Período 1

In [ ]:
start_date_train = pd.to_datetime('2025-09-26 00:00:00')
end_date_train = pd.to_datetime('2025-09-30 00:00:00')

mask = (df_dataset[timestamp] >= start_date_train) & (df_dataset[timestamp] <= end_date_train)
df_train = df_dataset.loc[mask]

# pre - processamento
print(df_train.shape)

list_periods_train = []
for pro in pre_process:
    df_train,_,list_aux = ppd.drop_transitorio_desligado(df_train,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train.reset_index(inplace=True, drop=True)

print(df_train.shape)

eixoX_train = df_train[timestamp]
df_train = df_train.drop(timestamp,axis=1)
df_train1 = df_train.copy()

df_train_list.append(df_train1)

## Período 2

In [ ]:
start_date_train2 = pd.to_datetime('2025-12-12 00:00:00')
end_date_train2 = pd.to_datetime('2025-12-12 04:00:00')


mask = (df_dataset[timestamp] > start_date_train2) & (df_dataset[timestamp] <= end_date_train2)
df_train2 = df_dataset.loc[mask]

# pre - processamento
print(df_train2.shape)

list_periods_train = []
for pro in pre_process:
    df_train2,_,list_aux = ppd.drop_transitorio_desligado(df_train2,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train2.reset_index(inplace=True, drop=True)

print(df_train2.shape)


eixoX_train2 = df_train2[timestamp]
df_train2 = df_train2.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train2], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train2], ignore_index=True)

df_train_list.append(df_train2)

## Período 3

In [ ]:
#Data train 
start_date_train3 = pd.to_datetime('2026-01-06 12:00:00') 
end_date_train3 = pd.to_datetime('2026-01-07 00:00:00')

mask = (df_dataset[timestamp] > start_date_train3) & (df_dataset[timestamp] <= end_date_train3)
df_train3 = df_dataset.loc[mask]

# pre - processamento
print(df_train3.shape)

list_periods_train = []
for pro in pre_process:
    df_train3,_,list_aux = ppd.drop_transitorio_desligado(df_train3,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train3.reset_index(inplace=True, drop=True)

print(df_train3.shape)


eixoX_train3 = df_train3[timestamp]
df_train3 = df_train3.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train3], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train3], ignore_index=True)

df_train_list.append(df_train3)

## Período 4

In [ ]:
start_date_train4 = pd.to_datetime('2026-01-10 06:00:00') 
end_date_train4 = pd.to_datetime('2026-01-10 18:00:00')

mask = (df_dataset[timestamp] > start_date_train4) & (df_dataset[timestamp] <= end_date_train4)
df_train4 = df_dataset.loc[mask]

# pre - processamento
print(df_train4.shape)

list_periods_train = []
for pro in pre_process:
    df_train4,_,list_aux = ppd.drop_transitorio_desligado(df_train4,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train4.reset_index(inplace=True, drop=True)

print(df_train4.shape)


eixoX_train4 = df_train4[timestamp]
df_train4 = df_train4.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train4], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train4], ignore_index=True)

df_train_list.append(df_train4)

## Período 5

In [ ]:
start_date_train5 = pd.to_datetime('2026-01-23 06:00:00') 
end_date_train5 = pd.to_datetime('2026-01-24 20:00:00')

mask = (df_dataset[timestamp] > start_date_train5) & (df_dataset[timestamp] <= end_date_train5)
df_train5 = df_dataset.loc[mask]

# pre - processamento
print(df_train5.shape)

list_periods_train = []
for pro in pre_process:
    df_train5,_,list_aux = ppd.drop_transitorio_desligado(df_train5,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train5.reset_index(inplace=True, drop=True)

print(df_train5.shape)


eixoX_train5 = df_train5[timestamp]
df_train5 = df_train5.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train5], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train5], ignore_index=True)

df_train_list.append(df_train5)

## Período 6

In [ ]:
start_date_train6 = pd.to_datetime('2026-02-11 22:00:00') 
end_date_train6 = pd.to_datetime('2026-02-12 00:00:00')

mask = (df_dataset[timestamp] > start_date_train6) & (df_dataset[timestamp] <= end_date_train6)
df_train6 = df_dataset.loc[mask]

# pre - processamento
print(df_train6.shape)

list_periods_train = []
for pro in pre_process:
    df_train6,_,list_aux = ppd.drop_transitorio_desligado(df_train6,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train6.reset_index(inplace=True, drop=True)

print(df_train6.shape)


eixoX_train6 = df_train6[timestamp]
df_train6 = df_train6.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train6], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train6], ignore_index=True)

df_train_list.append(df_train6)

## Período 7

In [ ]:
start_date_train7 = pd.to_datetime('2026-03-02 10:00:00') 
end_date_train7 = pd.to_datetime('2026-03-02 14:00:00')

mask = (df_dataset[timestamp] > start_date_train7) & (df_dataset[timestamp] <= end_date_train7)
df_train7 = df_dataset.loc[mask]

# pre - processamento
print(df_train7.shape)

list_periods_train = []
for pro in pre_process:
    df_train7,_,list_aux = ppd.drop_transitorio_desligado(df_train7,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train7.reset_index(inplace=True, drop=True)

print(df_train7.shape)


eixoX_train7 = df_train7[timestamp]
df_train7 = df_train7.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train7], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train7], ignore_index=True)

df_train_list.append(df_train7)

## Período 8

In [ ]:
start_date_train8 = pd.to_datetime('2026-03-12 00:00:00') 
end_date_train8 = pd.to_datetime('2026-03-12 12:00:00')

mask = (df_dataset[timestamp] > start_date_train8) & (df_dataset[timestamp] <= end_date_train8)
df_train8 = df_dataset.loc[mask]

# pre - processamento
print(df_train8.shape)

list_periods_train = []
for pro in pre_process:
    df_train8,_,list_aux = ppd.drop_transitorio_desligado(df_train8,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train8.reset_index(inplace=True, drop=True)

print(df_train8.shape)


eixoX_train8 = df_train8[timestamp]
df_train8 = df_train8.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train8], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train8], ignore_index=True)

df_train_list.append(df_train8)

## Período 9 TESTE

In [ ]:
#Data train 
start_date_train9 = pd.to_datetime('2026-03-15 00:00:00')
end_date_train9 = pd.to_datetime('2026-03-17 00:00:00')

mask = (df_dataset[timestamp] > start_date_train9) & (df_dataset[timestamp] <= end_date_train9)
df_train9 = df_dataset.loc[mask]

# pre - processamento
print(df_train9.shape)

list_periods_train = []
for pro in pre_process:
    df_train9,_,list_aux = ppd.drop_transitorio_desligado(df_train9,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_train = [*list_periods_train,*list_aux]

df_train9.reset_index(inplace=True, drop=True)

print(df_train9.shape)


eixoX_train9 = df_train9[timestamp]
df_train9 = df_train9.drop(timestamp,axis=1)

df_train = pd.concat([df_train, df_train9], ignore_index=True)
eixoX_train = pd.concat([eixoX_train, eixoX_train9], ignore_index=True)

df_train_list.append(df_train9)

## Criação do Modelo

In [ ]:
mscvae = MSCVAE()

gain = 0.7
epochs = 20
mscvae.fit(df_train_list, gain=gain, epochs=epochs, verbose=True)

In [ ]:
mscred = MSCRED()

gain = 1
epochs = 20
mscred.fit(df_train_list, gain=gain, epochs=epochs)

In [ ]:
ann_ae = ANN_AE()

gain = 1
epochs = 20
ann_ae.fit(df_train_list, epochs=epochs, gain=gain)

In [ ]:
lstm_ae = LSTM_AE()

gain = 1
epochs = 20
lstm_ae.fit(df_train_list, epochs=epochs, gain=gain)

In [ ]:
cnn_ae = CNN_AE()

gain = 1
epochs = 20
cnn_ae.fit(df_train_list, epochs=epochs, gain=gain)

# Seleção do período para testar o modelo

In [ ]:
start_date = pd.to_datetime("2026-01-01 00:00:00")
end_date = pd.to_datetime("2026-05-01 00:00:00")

mask = (df_dataset[timestamp] > start_date) & (df_dataset[timestamp] <= end_date)
df_test = df_dataset.loc[mask]

list_periods_nan = ppd.select_periods_nan(df_dataset,timestamp)

# pre - processamento
print(df_test.shape)

list_periods_test = []
for pro in pre_process:
    df_test,_,list_aux = ppd.drop_transitorio_desligado(df_test,pro["variable_off"],pro["limit_off"],pro["interval_off"],timestamp,pre_corte=pro["pre_cut"],pos_corte=pro["after_cut"])
    list_periods_test = [*list_periods_test,*list_aux]
    
print(df_test.shape)

df_test_aux = df_test.copy()
eixoX_test = df_test[timestamp]
df_test = df_test.drop(timestamp,axis=1)

df_test.reset_index(inplace=True, drop=True)

# Predição

In [ ]:
predictions_mscvae = mscvae.predict(df_test, timestamps=eixoX_test)

In [ ]:
predictions_mscred = mscred.predict(df_test, timestamps=eixoX_test)

In [ ]:
predictions_ann_ae = ann_ae.predict(df_test, timestamps=eixoX_test)

In [ ]:
predictions_lstm_ae = lstm_ae.predict(df_test, timestamps=eixoX_test)

In [ ]:
predictions_cnn_ae = cnn_ae.predict(df_test, timestamps=eixoX_test)

## Gráfico de Detecção

In [ ]:
list_periods_test = ppd.merge_periods(list_periods_test)
fig_all_period = utils.graph_predict(predictions_mscvae["phi"]/(mscvae.threshold), predictions_mscvae["timestamp"], mscvae.threshold/(mscvae.threshold), None,start_date, end_date, list_periods=None, plot_anomalies=False)
fig_all_period.show()

In [ ]:
list_periods_test = ppd.merge_periods(list_periods_test)
fig_all_period = utils.graph_predict(predictions_mscred["phi"]/(mscred.threshold), predictions_mscred["timestamp"], mscred.threshold/(mscred.threshold), None,start_date, end_date, list_periods=None, plot_anomalies=False)
fig_all_period.show()

In [ ]:
list_periods_test = ppd.merge_periods(list_periods_test)
fig_all_period = utils.graph_predict(predictions_ann_ae["phi"]/(ann_ae.threshold), predictions_ann_ae["timestamp"], ann_ae.threshold/(ann_ae.threshold), None,start_date, end_date, list_periods=None, plot_anomalies=False)
fig_all_period.show()

In [ ]:
list_periods_test = ppd.merge_periods(list_periods_test)
fig_all_period = utils.graph_predict(predictions_lstm_ae["phi"]/(lstm_ae.threshold), predictions_lstm_ae["timestamp"], lstm_ae.threshold/(lstm_ae.threshold), None,start_date, end_date, list_periods=None, plot_anomalies=False)
fig_all_period.show()

In [ ]:
list_periods_test = ppd.merge_periods(list_periods_test)
fig_all_period = utils.graph_predict(predictions_cnn_ae["phi"]/(cnn_ae.threshold), predictions_cnn_ae["timestamp"], cnn_ae.threshold/(cnn_ae.threshold), None,start_date, end_date, list_periods=None, plot_anomalies=False)
fig_all_period.show()

# Contribuição

In [ ]:
contribution, df_reconstruction = mscvae.contribution(df_test, df_sistema)
pd.DataFrame().from_dict(contribution).head(10)

In [ ]:
contribution, df_reconstruction = mscred.contribution(df_test, df_sistema)
pd.DataFrame().from_dict(contribution).head(10)

In [ ]:
contribution, df_reconstruction = ann_ae.contribution(df_test, df_sistema=df_sistema)
pd.DataFrame().from_dict(contribution).head(10)

In [ ]:
contribution, df_reconstruction = lstm_ae.contribution(df_test, df_sistema=df_sistema)
pd.DataFrame().from_dict(contribution).head(10)

In [ ]:
contribution, df_reconstruction = cnn_ae.contribution(df_test, df_sistema=df_sistema)
pd.DataFrame().from_dict(contribution).head(10)

## Gráfico Univariado

In [ ]:
mask = (df_dataset[timestamp] > start_date) & (df_dataset[timestamp] <= end_date)
df_aux = df_dataset.loc[mask]

eixoX_aux = df_aux[timestamp]
df_aux = df_aux.drop(timestamp,axis=1)

df_aux.reset_index(inplace=True, drop=True)
df_aux.head()

lista = ['392P2007.03HV.RE.LA']
fig = utils.graph_variables(df_aux, eixoX_aux,lista, df_projection=df_reconstruction)

fig.show()

In [ ]:
lista = ['392P2007A_TT']
fig = utils.graph_variables(df_aux, eixoX_aux,lista, df_projection=df_reconstruction)

fig.show()

In [ ]:
lista = ['392P2007.05HV.BM.LA']
fig = utils.graph_variables(df_aux, eixoX_aux,lista, df_projection=df_reconstruction)

fig.show()

In [ ]:
lista = ['392P2007.05HE2.BM.LA']
fig = utils.graph_variables(df_aux, eixoX_aux,lista, df_projection=df_reconstruction)

fig.show()

In [ ]:
# fig.write_html("output/Univar_"+process_name+".html")